In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import librosa
import librosa.display
import warnings
warnings.filterwarnings('ignore')

# ========================= CONFIGURATION =========================
# Update these paths according to your Kaggle setup

# Option 1: If you uploaded model as a Kaggle Dataset
# MODEL_PATH = '/kaggle/input/your-model-name/music_genre_classifier.keras'

# Option 2: If model is in working directory
# MODEL_PATH = '/kaggle/working/music_genre_classifier.keras'

# Option 3: If model is in input directory
# MODEL_PATH = '/kaggle/input/models/seeshcore/music-genreclassifier/keras/default/1/model.keras'

# Option 4: Try to find the model automatically
def find_model_path():
    """Automatically find the model file"""
    possible_paths = [
        '/kaggle/input/models/seeshcore/music-genreclassifier/keras/default/1/model.keras',
        '/kaggle/input/models/seeshcore/music-genreclassifier/keras/default/1/model.h5',
        '/kaggle/working/music_genre_classifier.keras',
        '/kaggle/working/music_genre_classifier.h5',
        'music_genre_classifier.keras',
        'music_genre_classifier.h5',
        '/kaggle/input/your-model/music_genre_classifier.keras',
    ]
    
    # Also check for any .keras or .h5 files in input directory
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            for file in files:
                if file.endswith(('.keras', '.h5')):
                    possible_paths.append(os.path.join(root, file))
    
    for path in possible_paths:
        if os.path.exists(path):
            return path
    
    return None

MODEL_PATH = find_model_path()

if MODEL_PATH is None:
    print("❌ Model not found! Please update MODEL_PATH.")
    print("Available files in /kaggle/input:")
    if os.path.exists('/kaggle/input'):
        for item in os.listdir('/kaggle/input'):
            print(f"  - {item}")
            if os.path.isdir(f'/kaggle/input/{item}'):
                subpath = f'/kaggle/input/{item}'
                for sub in os.listdir(subpath):
                    print(f"    - {sub}")
    exit(1)

print(f"✅ Using model: {MODEL_PATH}")

# GTZAN dataset path
DATASET_PATHS = [
    '/kaggle/input/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/Data/genres_original',
    '/kaggle/input/gtzan-dataset-music-genre-classification/Data/genres_original',
    '/kaggle/input/gtzan-dataset-music-genre-classification/genres_original',
    './genres_original'
]

DATASET_ROOT = None
for path in DATASET_PATHS:
    if os.path.exists(path):
        DATASET_ROOT = path
        break

if DATASET_ROOT is None:
    print("❌ Dataset not found! Please add GTZAN dataset.")
    exit(1)

print(f"✅ Using dataset: {DATASET_ROOT}")

# Number of random files to test
NUM_SAMPLES = 5

# Genres (must match the order your model was trained on)
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop',
          'jazz', 'metal', 'pop', 'reggae', 'rock']

# Audio preprocessing parameters
SAMPLE_RATE = 22050
DURATION = 30  # seconds
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
# ================================================================

def get_model_input_shape(model):
    """Get the expected input shape of the model"""
    input_shape = model.input_shape
    print(f"Model input shape: {input_shape}")
    return input_shape

def preprocess_audio_mfcc(file_path):
    """Extract MFCC features (common for DNN models)"""
    try:
        # Load audio
        y, sr = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION)
        
        # Pad or trim to exact duration
        if len(y) < SAMPLE_RATE * DURATION:
            y = np.pad(y, (0, SAMPLE_RATE * DURATION - len(y)))
        else:
            y = y[:SAMPLE_RATE * DURATION]
        
        # Extract MFCC features
        mfccs = librosa.feature.mfcc(
            y=y, 
            sr=sr, 
            n_mfcc=13,
            n_fft=N_FFT, 
            hop_length=HOP_LENGTH
        )
        
        # Take mean across time
        mfccs_mean = np.mean(mfccs.T, axis=0)
        
        return mfccs_mean
        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

def preprocess_audio_mel(file_path):
    """Extract Mel-spectrogram features (common for CNN models)"""
    try:
        # Load audio
        y, sr = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION)
        
        # Pad or trim to exact duration
        if len(y) < SAMPLE_RATE * DURATION:
            y = np.pad(y, (0, SAMPLE_RATE * DURATION - len(y)))
        else:
            y = y[:SAMPLE_RATE * DURATION]
        
        # Extract Mel spectrogram
        mel_spec = librosa.feature.melspectrogram(
            y=y, 
            sr=sr, 
            n_fft=N_FFT, 
            hop_length=HOP_LENGTH, 
            n_mels=N_MELS
        )
        
        # Convert to log scale (dB)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        
        # Normalize
        mel_spec_db = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min() + 1e-8)
        
        # Add channel dimension
        mel_spec_db = mel_spec_db[..., np.newaxis]  # shape: (128, time_frames, 1)
        
        # Resize if needed (some models expect fixed size)
        # Uncomment if your model expects fixed shape
        # if mel_spec_db.shape[1] != 128:
        #     mel_spec_db = tf.image.resize(mel_spec_db, (N_MELS, 128)).numpy()
        
        return mel_spec_db
        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

def preprocess_audio_combined(file_path):
    """Extract combined features (MFCC + Mel + Chroma)"""
    try:
        # Load audio
        y, sr = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION)
        
        # Pad or trim
        if len(y) < SAMPLE_RATE * DURATION:
            y = np.pad(y, (0, SAMPLE_RATE * DURATION - len(y)))
        else:
            y = y[:SAMPLE_RATE * DURATION]
        
        # MFCCs
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, n_fft=N_FFT, hop_length=HOP_LENGTH)
        mfccs_mean = np.mean(mfccs.T, axis=0)
        
        # Mel-spectrogram
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        mel_spec_mean = np.mean(mel_spec_db.T, axis=0)
        
        # Chroma
        chroma = librosa.feature.chroma_stft(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
        chroma_mean = np.mean(chroma.T, axis=0)
        
        # Combine
        features = np.concatenate([mfccs_mean, mel_spec_mean, chroma_mean])
        
        return features
        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

def predict_with_auto_feature(file_path, model):
    """
    Try different feature extraction methods and use the one that works
    """
    input_shape = model.input_shape
    
    # Try different feature extraction methods
    methods = [
        ('MFCC', preprocess_audio_mfcc, (13,)),
        ('Mel-Spectrogram', preprocess_audio_mel, (128, None, 1)),
        ('Combined', preprocess_audio_combined, None)
    ]
    
    for method_name, method_func, expected_shape in methods:
        try:
            features = method_func(file_path)
            if features is None:
                continue
            
            # Check if features match model input shape
            if len(input_shape) == 2:  # Dense model
                if features.ndim == 1:
                    features = features.reshape(1, -1)
                    predictions = model.predict(features, verbose=0)
                    return predictions, method_name
                    
            elif len(input_shape) == 4:  # CNN model (batch, height, width, channels)
                if features.ndim == 3:
                    features = np.expand_dims(features, axis=0)
                    predictions = model.predict(features, verbose=0)
                    return predictions, method_name
                    
            else:  # Try any shape
                if features.ndim == 1:
                    features = features.reshape(1, -1)
                elif features.ndim == 3:
                    features = np.expand_dims(features, axis=0)
                
                try:
                    predictions = model.predict(features, verbose=0)
                    return predictions, method_name
                except:
                    continue
                    
        except Exception as e:
            continue
    
    return None, None

def main():
    # Load the model
    print("\n" + "="*60)
    print("🎵 Loading Model...")
    print("="*60)
    
    try:
        model = load_model(MODEL_PATH)
        print("✅ Model loaded successfully!")
        print(f"Model type: {type(model)}")
        print(f"Model input shape: {model.input_shape}")
        print(f"Model output shape: {model.output_shape}")
        
        # Show model summary
        print("\n📊 Model Architecture:")
        model.summary()
        
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        return
    
    # Get all .wav files
    print("\n" + "="*60)
    print("📁 Finding Audio Files...")
    print("="*60)
    
    audio_files = []
    for genre in GENRES:
        genre_dir = os.path.join(DATASET_ROOT, genre)
        if os.path.exists(genre_dir):
            files = [os.path.join(genre_dir, f) for f in os.listdir(genre_dir) 
                    if f.endswith('.wav')]
            audio_files.extend(files)
            print(f"Found {len(files)} files in {genre}")
    
    if not audio_files:
        print("❌ No audio files found! Check DATASET_ROOT path.")
        print(f"Current DATASET_ROOT: {DATASET_ROOT}")
        return
    
    print(f"\n✅ Total audio files found: {len(audio_files)}")
    
    # Pick random files
    random_files = random.sample(audio_files, min(NUM_SAMPLES, len(audio_files)))
    
    print("\n" + "="*60)
    print("🎯 Testing Predictions")
    print("="*60)
    
    results = []
    
    for idx, file_path in enumerate(random_files, 1):
        true_genre = os.path.basename(os.path.dirname(file_path))
        
        print(f"\n[{idx}/{len(random_files)}] Testing: {os.path.basename(file_path)}")
        
        # Try different feature extraction methods
        predictions, method_used = predict_with_auto_feature(file_path, model)
        
        if predictions is None:
            print("   ❌ Could not process file with any feature extraction method")
            continue
        
        predicted_idx = np.argmax(predictions[0])
        predicted_genre = GENRES[predicted_idx]
        confidence = predictions[0][predicted_idx] * 100
        
        results.append({
            'file': os.path.basename(file_path),
            'true_genre': true_genre,
            'predicted_genre': predicted_genre,
            'confidence': confidence,
            'correct': true_genre == predicted_genre,
            'method': method_used
        })
        
        # Print result
        status = "✅" if true_genre == predicted_genre else "❌"
        print(f"   {status} True: {true_genre:12} | Predicted: {predicted_genre:12} | Confidence: {confidence:.2f}%")
        print(f"   Feature method: {method_used}")
        
        # Print top 3 predictions
        top_indices = np.argsort(predictions[0])[-3:][::-1]
        print("   Top 3 predictions:")
        for idx_pred in top_indices:
            genre = GENRES[idx_pred]
            prob = predictions[0][idx_pred] * 100
            print(f"      {genre:12}: {prob:.2f}%")
    
    # Summary
    if results:
        print("\n" + "="*60)
        print("📊 Summary")
        print("="*60)
        
        correct = sum(1 for r in results if r['correct'])
        total = len(results)
        accuracy = correct / total * 100 if total > 0 else 0
        
        print(f"Total tests: {total}")
        print(f"Correct: {correct}")
        print(f"Accuracy: {accuracy:.2f}%")
        
        # Show results table
        print("\n📋 Results:")
        print("-" * 80)
        print(f"{'File':<25} {'True':<12} {'Predicted':<12} {'Confidence':<10} {'Status':<8}")
        print("-" * 80)
        for r in results:
            status = "✅" if r['correct'] else "❌"
            print(f"{r['file'][:24]:<25} {r['true_genre']:<12} {r['predicted_genre']:<12} {r['confidence']:<10.2f}% {status}")
        print("-" * 80)

if __name__ == "__main__":
    main()

2026-06-29 17:35:55.691900: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782754555.716530     267 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782754555.724250     267 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782754555.742524     267 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782754555.742543     267 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782754555.742545     267 computation_placer.cc:177] computation placer alr

✅ Using model: /kaggle/input/models/seeshcore/music-genreclassifier/keras/default/1/music_genre_classifier.keras
✅ Using dataset: /kaggle/input/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/Data/genres_original

🎵 Loading Model...


I0000 00:00:1782754561.822522     267 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782754561.827961     267 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


✅ Model loaded successfully!
Model type: <class 'keras.src.models.sequential.Sequential'>
Model input shape: (None, 13)
Model output shape: (None, 10)

📊 Model Architecture:


Model: "sequential_1"

...

 Non-trainable params: 896 (3.50 KB)

 Optimizer params: 46,284 (180.80 KB)


📁 Finding Audio Files...
Found 100 files in blues
Found 100 files in classical
Found 100 files in country
Found 100 files in disco
Found 100 files in hiphop
Found 100 files in jazz
Found 100 files in metal
Found 100 files in pop
Found 100 files in reggae
Found 100 files in rock

✅ Total audio files found: 1000

🎯 Testing Predictions

[1/5] Testing: metal.00092.wav


I0000 00:00:1782754564.711228     330 service.cc:152] XLA service 0x7bd3d0003fa0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782754564.711263     330 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1782754564.711266     330 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1782754564.776406     330 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1782754565.026356     330 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


   ❌ True: metal        | Predicted: rock         | Confidence: 58.84%
   Feature method: MFCC
   Top 3 predictions:
      rock        : 58.84%
      country     : 18.03%
      blues       : 9.92%

[2/5] Testing: hiphop.00077.wav
   ❌ True: hiphop       | Predicted: pop          | Confidence: 48.84%
   Feature method: MFCC
   Top 3 predictions:
      pop         : 48.84%
      disco       : 35.41%
      hiphop      : 9.49%

[3/5] Testing: pop.00033.wav
   ❌ True: pop          | Predicted: disco        | Confidence: 74.56%
   Feature method: MFCC
   Top 3 predictions:
      disco       : 74.56%
      rock        : 10.20%
      pop         : 9.76%

[4/5] Testing: rock.00049.wav
   ❌ True: rock         | Predicted: reggae       | Confidence: 23.98%
   Feature method: MFCC
   Top 3 predictions:
      reggae      : 23.98%
      hiphop      : 21.00%
      country     : 17.84%

[5/5] Testing: classical.00010.wav
   ✅ True: classical    | Predicted: classical    | Confidence: 92.00%
   Feature

In [ ]:
#pip install streamlit tensorflow numpy librosa